<a href="https://colab.research.google.com/github/cristianzucconi2-web/iatoarts/blob/main/iatorartsFinale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import numpy as np
from deap import base, creator, tools, algorithms
from music21 import stream, note, chord, duration, instrument, midi

# --- CONFIGURAZIONE ---
NUM_GENERATIONS = 100
POP_SIZE = 50
SONG_LENGTH_MEASURES = 64  # Circa 2-3 minuti a seconda del tempo
GENOME_SIZE = SONG_LENGTH_MEASURES * 4 # 4 sedicesimi per battuta

# --- LOGICA DI FITNESS (Il Giudice) ---
def evaluate_melody(individual):
    penalty = 0
    consonant_intervals = [0, 3, 4, 5, 7, 8, 9, 12] # Unisono, terze, quarte, quinte...

    # 1. Penalità per salti troppo grandi (Fluidità)
    for i in range(len(individual) - 1):
        diff = abs(individual[i] - individual[i+1])
        if diff > 7: # Più di una quinta
            penalty += 10
        if diff == 0: # Troppe note ripetute sono noiose
            penalty += 2

    # 2. Premio per tonalità (es. Do Maggiore)
    c_major_scale = [0, 2, 4, 5, 7, 9, 11]
    for pitch in individual:
        if pitch % 12 not in c_major_scale:
            penalty += 15

    return penalty,

# --- SETUP EVOLUTIVO ---
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
# Note tra 60 (Do centrale) e 84 (Do due ottave sopra)
toolbox.register("attr_note", random.randint, 60, 84)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_note, n=GENOME_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evaluate_melody)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=2, indpb=0.1)
toolbox.register("select", tools.selTournament, tournsize=3)

# --- ESECUZIONE ---
def run_evolution():
    pop = toolbox.population(n=POP_SIZE)
    hof = tools.HallOfFame(1)

    algorithms.eaSimple(pop, toolbox, cxpb=0.7, mutpb=0.2, ngen=NUM_GENERATIONS,
                        halloffame=hof, verbose=True)

    return hof[0]

# --- CONVERSIONE IN MUSICA (Polifonia e Strumenti) ---
def create_midi(best_genome):
    s = stream.Score()

    # Creiamo due tracce: Piano e Violino
    piano_part = stream.Part()
    piano_part.insert(0, instrument.Piano())

    violin_part = stream.Part()
    violin_part.insert(0, instrument.Violin())

    for i, p in enumerate(best_genome):
        n = note.Note(p)
        n.quarterLength = 0.5 # Ottavi

        # Logica di "Orchestrazione" semplice:
        # Alterniamo o suoniamo insieme in base all'indice
        if i % 8 < 4:
            piano_part.append(n)
        elif i % 8 < 6:
            violin_part.append(n)
        else:
            # Suonano insieme (creiamo un clone per la seconda parte)
            piano_part.append(note.Note(p))
            violin_part.append(note.Note(p+12)) # Violino un'ottava sopra

    s.insert(0, piano_part)
    s.insert(0, violin_part)

    s.write('midi', fp='evoluzione_musicale.mid')
    print("File MIDI generato: evoluzione_musicale.mid")

if __name__ == "__main__":
    best = run_evolution()
    create_midi(best)

ModuleNotFoundError: No module named 'deap'